In [40]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [48]:
# Generate X and Y datasets
X = []
Y = []
for i in np.linspace(0.1, 1, 21):
    x = np.zeros((5, 4))
    y = np.zeros((5, 4))
    x[:, 0] = i
    X.append(x.T)
    y[:, :] = i
    Y.append(y.T)

X = np.array(X)
X = torch.tensor(X, dtype=torch.float32)  # Shape: (21, 4, 5)
Y = np.array(Y)
Y = torch.tensor(Y, dtype=torch.float32)  # Shape: (21, 4, 5)

# Check the shapes of input and output tensors
print("X shape:", X.shape)
print("Y shape:", Y.shape)
print(X[0])
print(Y[0])

X shape: torch.Size([21, 4, 5])
Y shape: torch.Size([21, 4, 5])
tensor([[0.1000, 0.1000, 0.1000, 0.1000, 0.1000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000]])
tensor([[0.1000, 0.1000, 0.1000, 0.1000, 0.1000],
        [0.1000, 0.1000, 0.1000, 0.1000, 0.1000],
        [0.1000, 0.1000, 0.1000, 0.1000, 0.1000],
        [0.1000, 0.1000, 0.1000, 0.1000, 0.1000]])


In [62]:
# Define the LSTM model
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # LSTM layer
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        
        # Fully connected layer
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # Initialize hidden and cell states
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        
        # Forward propagate LSTM
        out, _ = self.lstm(x, (h0, c0))
        
        # Pass through fully connected layer
        out = self.fc(out)
        return out

# Define hyperparameters
input_size = X.size(-1)  # 5 (each time step has 5 features)
hidden_size = 50
output_size = Y.size(-1)  # 5
num_layers = 1
num_epochs = 1000
learning_rate = 0.001

# Instantiate the model, define the loss function and optimizer
model = LSTMModel(input_size, hidden_size, output_size, num_layers)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [63]:
# Training loop
for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0

    for i in range(X.size(0)):
        # Get each input and target pair individually
        single_input = X[i].unsqueeze(0)  # Shape: (1, 4, 5)
        single_output = Y[i].unsqueeze(0)  # Shape: (1, 4, 5)

        # Forward pass
        outputs = model(single_input)
        loss = criterion(outputs, single_output)
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    
    # Print average loss for each epoch
    if (epoch + 1) % 100 == 0:
        avg_loss = total_loss / X.size(0)
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.6f}')

Epoch [100/1000], Loss: 0.000191
Epoch [200/1000], Loss: 0.000074
Epoch [300/1000], Loss: 0.000061
Epoch [400/1000], Loss: 0.000016
Epoch [500/1000], Loss: 0.000050
Epoch [600/1000], Loss: 0.000291
Epoch [700/1000], Loss: 0.000012
Epoch [800/1000], Loss: 0.000149
Epoch [900/1000], Loss: 0.000098
Epoch [1000/1000], Loss: 0.000092


In [64]:
with torch.no_grad():
    predictions = []
    i = 5
    single_input = X[i].unsqueeze(0)  # Add batch dimension
    predicted_output = model(single_input)
    print("Actual:\n", Y[i])
    print("Prediction:\n", predicted_output)


Actual:
 tensor([[0.3250, 0.3250, 0.3250, 0.3250, 0.3250],
        [0.3250, 0.3250, 0.3250, 0.3250, 0.3250],
        [0.3250, 0.3250, 0.3250, 0.3250, 0.3250],
        [0.3250, 0.3250, 0.3250, 0.3250, 0.3250]])
Prediction:
 tensor([[[0.3171, 0.3183, 0.3170, 0.3171, 0.3170],
         [0.3195, 0.3207, 0.3193, 0.3185, 0.3186],
         [0.3217, 0.3237, 0.3220, 0.3215, 0.3215],
         [0.3243, 0.3258, 0.3250, 0.3246, 0.3251]]])
